# Historical notebook ? superseded
Source retained for provenance only. Outputs cleared. Do not run: this workflow accesses 2025 outcomes and uses superseded definitions. Use ../06_major_outage_eda.ipynb and ../../docs/dataset_card.md instead.

In [ ]:
raise RuntimeError('Historical notebook disabled: use notebooks/06_major_outage_eda.ipynb; do not explore 2025 outcomes.')

# 01 - EAGLE-I outage data quality

What values we have, what is null or missing, and anything that looks wrong.
Data: `eaglei/outages_YYYY.parquet` (customers out per county, every 15 min), plus the
EAGLE-I reference files (MCC customer counts, coverage history, DQI).

Each year is 0.4-6M rows, so we go one year at a time and keep only small summaries.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import (load_outages, load_customer_counts, load_census, load_coverage_history,
                      load_dqi)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. One pass over every year

In [ ]:
# Small summaries collected per year; the full year is dropped before loading the next
summary, per_county, monthly, nulls, schemas = [], [], [], [], {}

for y in C.YEARS:
    df = load_outages([y])
    out = df["customers_out"]
    t = df["run_start_time"]
    slots = (t.max() - t.min()) / pd.Timedelta("15min") + 1   # possible 15-min readings in this file
    schemas[y] = tuple(df.dtypes.astype(str).items())         # (column, dtype) pairs, compared in section 2

    # Year-level counts
    summary.append({
        "year": y,
        "rows": len(df),
        "counties": df["fips_code"].nunique(),
        "first": t.min(),
        "last": t.max(),
        "minutes_seen": sorted(t.dt.minute.unique()),
        "odd_minute": (t.dt.minute % 15 != 0).sum(),           # stamped off the 15-min grid
        "dec30_rows": ((t.dt.month == 12) & (t.dt.day == 30)).sum(),
        "dec31_rows": ((t.dt.month == 12) & (t.dt.day == 31)).sum(),
        "null_customers_out": out.isna().sum(),
        "null_time": t.isna().sum(),
        "null_fips": df["fips_code"].isna().sum(),
        "zeros": (out == 0).sum(),
        "negative": (out < 0).sum(),
        "non_integer": (out.dropna() % 1 != 0).sum(),
        "dup_county_time": df.duplicated(["fips_code", "run_start_time"]).sum(),
        "max_out": out.max(),
    })

    # Per county: how many readings, how many null, biggest outage
    c = df.groupby(["fips_code", "county", "state"], dropna=False).agg(
        readings=("customers_out", "size"),
        nulls=("customers_out", lambda s: s.isna().sum()),
        max_out=("customers_out", "max"),
    ).reset_index()
    c["year"] = y
    c["pct_of_possible"] = 100 * c["readings"] / slots
    per_county.append(c)

    # Per month: rows and counties reporting
    m = df.groupby(t.dt.month).agg(rows=("fips_code", "size"), counties=("fips_code", "nunique"))
    m["year"] = y
    monthly.append(m.reset_index(names="month"))

    # Null customers_out by state and month
    n = df[out.isna()]
    nulls.append(n.groupby([n["state"], n["run_start_time"].dt.month.rename("month")]).size()
                  .rename("null_rows").reset_index().assign(year=y))

summary = pd.DataFrame(summary)
per_county = pd.concat(per_county, ignore_index=True)
monthly = pd.concat(monthly, ignore_index=True)
nulls = pd.concat(nulls, ignore_index=True)
summary

## 2. Columns: same types every year? Example values (2024)

In [ ]:
print("same columns and dtypes in every year:", len(set(schemas.values())) == 1, "\n")

df = load_outages([2024])
print(df.dtypes, "\n")
print(df.head(), "\n")
print(df["customers_out"].describe().round(1))

**How big are the values?** Most readings are a handful of customers; large values are rare.

In [ ]:
bins = [0, 1, 10, 100, 1_000, 10_000, 100_000, 10_000_000]
print(pd.cut(df["customers_out"], bins, right=False).value_counts().sort_index())

## 3. Missing readings (rows that are not there at all)

A full year has 35,040 15-min readings per county (35,136 in a leap year). EAGLE-I seems to
skip the row when nothing is out, so most counties have far fewer rows. `pct_of_possible` is
the share of 15-min slots that have a row.

In [ ]:
per_county.groupby("year")["pct_of_possible"].describe().round(1)

In [ ]:
per_county.boxplot(column="pct_of_possible", by="year", figsize=(10, 4))
plt.title("Share of 15-min slots with a row, per county")
plt.suptitle("")
plt.ylabel("% of possible readings")
plt.show()

**Counties reporting each month.** Gaps here are whole months where the data is thin.

In [ ]:
grid = monthly.pivot(index="year", columns="month", values="counties")
plt.figure(figsize=(10, 4))
plt.imshow(grid, aspect="auto", cmap="viridis")
plt.colorbar(label="counties with any row")
plt.xticks(range(12), range(1, 13))
plt.yticks(range(len(grid)), grid.index)
plt.xlabel("month")
plt.title("Counties reporting per month")
plt.show()
grid

**Last day of the year.** In 2014-2021 the files stop at Dec 31 00:00, so Dec 31 is missing.

In [ ]:
summary[["year", "last", "dec30_rows", "dec31_rows"]]

**Odd timestamps.** A few rows are stamped at :16, :31, :35 or :46 instead of 15-min marks.

In [ ]:
summary.loc[summary["odd_minute"] > 0, ["year", "minutes_seen", "odd_minute"]]

## 4. Null `customers_out` (row exists, value is empty)

In [ ]:
summary[["year", "rows", "null_customers_out"]].assign(
    pct_null=lambda d: (100 * d["null_customers_out"] / d["rows"]).round(2))

In [ ]:
nulls.pivot_table(index="year", columns="state", values="null_rows", aggfunc="sum", fill_value=0)

In [ ]:
nulls.pivot_table(index="year", columns="month", values="null_rows", aggfunc="sum", fill_value=0)

Are nulls a few counties that are always null, or scattered? (share of each county's rows that are null)

In [ ]:
has_null = per_county[per_county["nulls"] > 0].copy()
has_null["pct_null"] = 100 * has_null["nulls"] / has_null["readings"]
has_null.groupby("year")["pct_null"].describe().round(1)

## 5. Counties

Which study-area counties never show up, and does each FIPS code keep one name?

In [ ]:
census = load_census()
study = census.drop_duplicates("fips_code")[["fips_code", "county_name"]]
seen = set(per_county["fips_code"])
print(f"{len(seen)} of {len(study)} study-area counties appear in EAGLE-I at least once")
study[~study["fips_code"].isin(seen)]

In [ ]:
print("FIPS codes not in the census list:", sorted(seen - set(study["fips_code"])))
names = per_county.groupby("fips_code")[["county", "state"]].nunique()
names[(names > 1).any(axis=1)]

Counties per year by state, against the number of counties each state has.

In [ ]:
per_county.pivot_table(index="year", columns="state", values="fips_code", aggfunc="nunique")

## 6. Customer counts (MCC): the denominator for "% of customers out"

In [ ]:
mcc = load_customer_counts()
mcc = mcc[mcc["fips_code"].isin(study["fips_code"])]
print(f"MCC has {len(mcc)} of {len(study)} study counties")
study[~study["fips_code"].isin(mcc["fips_code"])]

County-years where the peak outage is more than the county's total customers (impossible, so a data problem or an MCC that is too low).

In [ ]:
peaks = per_county.merge(mcc[["fips_code", "Customers"]], on="fips_code", how="left")
peaks["peak_share"] = peaks["max_out"] / peaks["Customers"]
print(peaks["peak_share"].describe().round(3))
peaks[peaks["peak_share"] > 1].sort_values("peak_share", ascending=False)[
    ["year", "fips_code", "county", "state", "max_out", "Customers", "peak_share"]].head(20)

In [ ]:
print("county-years with peak > all customers, by state:")
peaks[peaks["peak_share"] > 1]["state"].value_counts()

Sanity check MCC against Census housing units: a county should have roughly 1 or more
customers per housing unit (homes plus businesses).

In [ ]:
housing = census[census["acs_year"] == 2023][["fips_code", "county_name", "housing_units"]]
check = housing.merge(mcc[["fips_code", "Customers"]], on="fips_code", how="left")
check["customers_per_home"] = check["Customers"] / check["housing_units"]
check["state"] = check["fips_code"].str[:2].map(C.STATES)
print(check.groupby("state")["customers_per_home"].agg(
    median="median", below_half=lambda s: (s < 0.5).sum(), counties="size").round(2))
check.sort_values("customers_per_home").head(15)

## 7. EAGLE-I's own coverage and data-quality scores

`coverage_history.csv`: share of each state's customers that EAGLE-I tracks (from 2018 on).

In [ ]:
cov = load_coverage_history()
cov = cov[cov["state"].isin(C.STATES.values())]
cov["year"] = pd.to_datetime(cov["year"], format="%m/%d/%y").dt.year
cov.pivot(index="year", columns="state", values="max_pct_covered")

`DQI.csv`: data quality index per FEMA region. All six study states are in **FEMA Region 4**.

In [ ]:
dqi = load_dqi()
dqi[dqi["fema"] == 4]

## 8. Summary

**Clean**
- Same 5 columns and types in all 12 years (this confirms the 2024 release matches).
- No null FIPS or timestamps, no negative or fractional counts, no duplicate county + time rows.
- Every FIPS code is a real study-area county and keeps one name.

**Problems to handle**
1. **North Carolina customer counts (MCC) are wrong.** 48 of 100 NC counties have under 0.5
   customers per home (Henderson: 24 customers for 57k homes); every other state is about 1.1-1.2.
   6 western NC counties have no MCC at all.
   -> For NC, use another denominator (e.g. Census housing units, or the county's highest outage seen).
2. **Some peaks are more than 100% of customers even where MCC looks fine** (123 county-years outside NC,
   248 in NC). Mostly hurricanes and small rural counties, e.g. Miami-Dade during Irma (2017): 1.78M out
   vs 951k customers. Early EAGLE-I likely double-counted some utilities. -> Cap peak share at 100% and flag these.
3. **Missing rows are mostly "nothing out".** Counties have rows for only about 15-33% of 15-min slots,
   and true zeros appear only from 2022 on. Before 2022 a missing row probably means 0.
4. **Nulls vs zeros changed over time.** 2017-2021 and 2025 have null `customers_out` (up to 8% of rows,
   spread over all states and months); 2022-2024 have zeros instead. Probably the same thing coded two ways;
   confirm before treating null as 0.
5. **Coverage grows**, mainly in Tennessee (36 -> 90 of 95 counties) and Alabama (22 -> 66 of 67).
   2014 only covers Nov-Dec. Five Tennessee counties never appear.
   EAGLE-I's own coverage is lowest in TN (55-75% of customers) and AL (about 80%).
6. **Dec 31 is missing in 2014-2021** (files stop at Dec 31 00:00).
7. A few hundred rows have odd minutes (:16, :31, :35, :46); round them to the nearest 15 min.